In [ ]:
from datetime import date , datetime

class GAME_DTO:
    price:int
    name:str
    source_url:str
    source_web:str
    register_time:date = datetime.now().date
    
    
    def to_dict(self):
        return self.__dict__


NameError: name 'DateTi' is not defined

In [ ]:
from abc import ABC, abstractmethod 
import re
### Orquestrador del scraping
class I_CONF(ABC):    
    
    @abstractmethod
    def buid_url(self, query: str) -> str:
        ...

    @property
    @abstractmethod
    def card_container_selector(self) -> str:
        ...
    
    
    @property
    @abstractmethod
    def source_web(self) -> str:
        ...
    
    
    @property
    @abstractmethod
    def next_page_selector(self) -> str:
        ...
    
    
    
    @property
    @abstractmethod
    def propietes_selector(self) -> dict[str, str]:
        ...
    
    @abstractmethod
    def create_dto(self)->object:
        ...
    
    
    def base_query_builder(self,query:str):
        query = re.sub(r"(?<=\S)\s+(?=\S)", "%20", query)  
        query = re.sub(r"\s+", "", query)
        return query;


    
    def make_dto(self,propertys:dict[str,str|float|int]):
        dto = self.create_dto()
        dto.source_web = self.source_web
        for key , value in propertys.items():
            setattr(dto, key, value)
        return dto
    

In [ ]:
class ENEBA_CONF(I_CONF):
    
    def buid_url(self, query):
        return "https://www.eneba.com/es/store/all?drms[]=steam&page=1&text=" + self.base_query_builder(query);
    
    
    @property
    def source_web(self) -> str:
        return "ENEBA"
    
    @property
    def card_container_selector(self)->str:
        return ".JZCH_t > .pFaGHa";
    
    @property
    def next_page_selector(self)->str:

        return "li.rc-pagination-next";
    
    
    @property
    def propietes_selector(self) -> dict[str, str]:
        return {
            "name" : "div.lirayz > span",
            "price":"div.VkR9uM > span > span",
            "source_url" : "div._vfaJQ > a"
        }
    
    def create_dto(self)->GAME_DTO:
        return GAME_DTO()

In [7]:
ENEBA_CONF().buid_url("dragon ball sparking zero")

'https://www.eneba.com/es/store/all?drms[]=steam&page=1&text=dragon%20ball%20sparking%20zero'

In [ ]:
from selenium.webdriver.chrome.webdriver import WebDriver as Chrome 
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.remote.webelement import WebElement
import time


class spider:
    def __init__(self, orq:I_CONF):
        self.__orq = orq

    def __create_nav(self):
        options = Options()
        options.add_argument("--headless=new")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("user-agent=Mozilla/5.0 ...")
        return Chrome(options=options)


    def __extractor(self, driver:Chrome):
        dtos = []
        while True:
            try:
                cards = self.__get_elements(driver,self.__orq.card_container_selector)
            
                dtos += [self.__extract_game_info(card) for card in cards]
                
                if not self.__next_page(driver):
                    break
            except EmptyResult as e:
                print(f"Scraping cordado: {e}")
                break
            
        return dtos
    
    def __extract_game_info(self, card: WebElement):
        game_info = {}
        
        for property , selector in self.__orq.propietes_selector.items():
            game_info[property] = self.__extract_transform(card,selector)
        return self.__orq.make_dto(game_info)
    
    
    

    def __extract_transform(self, card: WebElement, selector: str):
        element = self.__get_element(card, selector, 0.1)
        if element is None:
            raise EmptyResult(f"No se encontro {selector} en el card")

        tag_name = element.tag_name.lower()
        if tag_name == "a":
            href = element.get_attribute("href")
            if href:
                return href

        raw = element.text.strip()

        # Función auxiliar para limpiar y convertir a número
        def clean_and_convert_to_number(text):
            cleaned_text = text
            currency_symbols = ["€", "$", "£", "¥", "₹", "₽", "₩", "₺", "%", "-"]
            for symbol in currency_symbols:
                cleaned_text = cleaned_text.replace(symbol, "")
            cleaned_text = cleaned_text.strip()

            is_negative = False
            if cleaned_text.startswith("-") and cleaned_text[1:].replace(".", "", 1).replace(",", "", 1).isdigit():
                is_negative = True
                cleaned_text = cleaned_text[1:]

            if cleaned_text.isdigit():
                return int(cleaned_text) * (-1 if is_negative else 1)

            num_commas = cleaned_text.count(",")
            num_periods = cleaned_text.count(".")

            if num_commas == 1 and num_periods == 0:
                cleaned_text = cleaned_text.replace(",", ".")
            elif num_periods == 1 and num_commas == 0:
                pass
            elif num_commas > 0 and num_periods > 0:
                if cleaned_text.rfind(",") > cleaned_text.rfind("."):
                    cleaned_text = cleaned_text.replace(".", "")
                    cleaned_text = cleaned_text.replace(",", ".")
                else:
                    cleaned_text = cleaned_text.replace(",", "")

            try:
                return float(cleaned_text) * (-1 if is_negative else 1)
            except ValueError:
                return None

        number_value = clean_and_convert_to_number(raw)
        if number_value is not None:
            return number_value

        return raw


        
    
    
    
    def __go_to(self, driver, url: str):
        driver.get(url)
        self.__scroll_to_bottom(driver)
    
    def __next_page(self,driver:Chrome):
        time.sleep(0.5)
        try:
            button = self.__get_element(driver,self.__orq.next_page_selector,max_time=10)
            button.click()
            time.sleep(0.5)
            self.__scroll_to_bottom(driver)
            return True
        except:
            return False
        
        
    def __scroll_to_bottom(self, driver):
        time.sleep(0.5)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        
    
    def __get_elements(self, driver: Chrome, css_selector: str):
        wait = WebDriverWait(driver, 5)
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, css_selector)))
            return driver.find_elements(By.CSS_SELECTOR, css_selector)
        except:
            return []


    
    def __get_element(self,driver:Chrome|WebElement,css_selector:str,max_time=5):
        wait = WebDriverWait(driver, max_time)
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, css_selector)))
            return driver.find_element(By.CSS_SELECTOR,css_selector)
        except:
            return None

    def scraping_game(self, query: str):
        with self.__create_nav() as driver:
            url = self.__orq.buid_url(query)
            self.__go_to(driver,url)
            return self.__extractor(driver)
            
            
            
class EmptyResult(Exception):
    pass

In [55]:
spider = spider_game(ENEBA_CONF())


In [56]:
dtos:list[GAME_DTO] = spider.scraping_game("Dragon ball")
print(len(dtos))

Scraping cordado: No se encontro div.VkR9uM > span > span en el card
80


In [57]:
import pandas as pd


pd.DataFrame([dto.to_dict() for dto in dtos])

,source_web,name,price,source_url
0,ENEBA,DRAGON BALL: Sparking! ZERO (PC) Código de Ste...,27.01,https://www.eneba.com/es/steam-dragon-ball-spa...
1,ENEBA,Dragon Ball: Xenoverse 2 Código de Steam GLOBAL,4.84,https://www.eneba.com/es/steam-dragon-ball-xen...
2,ENEBA,DRAGON BALL XENOVERSE 2 Deluxe Edition (PC) St...,7.89,https://www.eneba.com/es/steam-dragon-ball-xen...
3,ENEBA,Dragon Ball Z: Kakarot Steam Key GLOBAL,11.37,https://www.eneba.com/es/steam-dragon-ball-z-k...
4,ENEBA,DLC\nDragon Ball Xenoverse 2 - Extra Pass (DLC...,11.37,https://www.eneba.com/es/steam-dragon-ball-xen...
...,...,...,...,...
75,ENEBA,DRAGON BALL: Sparking! ZERO (PC) Código de Ste...,38.90,https://www.eneba.com/es/steam-dragon-ball-spa...
76,ENEBA,DRAGON BALL: Sparking! ZERO - Ultimate Edition...,115.43,https://www.eneba.com/es/steam-dragon-ball-spa...
77,ENEBA,Dragon Ball Z: Kakarot Código de Steam UNITED ...,11.37,https://www.eneba.com/es/steam-dragon-ball-z-k...
78,ENEBA,DLC\nDragon Ball Xenoverse 2 - Extra Pass (DLC...,12.93,https://www.eneba.com/es/steam-dragon-ball-xen...
